In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
seed = 0
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [3]:
def ols(X, y):
  return np.linalg.inv(X.T @ X) @ X.T @ y

def mse(y, pred):
  return np.mean((y - pred) ** 2)

def rmse(y, pred):
  return np.sqrt(mse(y, pred))

def mae(y, pred):
  return np.mean(np.abs(y - pred))

def r_squared(y, pred):
  mean_y = np.mean(y)
  return 1 - (np.sum((y - pred) ** 2) / np.sum((y - mean_y) ** 2))

def adjusted_r_squared(y, pred, n_features):
  return 1 - (
    ((1 - r_squared(y, pred)) * (y.shape[0] - 1)) / 
    (y.shape[0] - n_features - 1)
  )


In [4]:
df['sex'] = df['sex'].cat.codes
df['smoker'] = df['smoker'].cat.codes
df['time'] = df['time'].cat.codes

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df):
  X = df.drop(columns=['tip', 'day'])
  y = df['tip'].to_numpy()

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
  X_train, X_validate, y_train, y_validate = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_validate_scaled = scaler.transform(X_validate)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_validate_scaled.shape == X_validate.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_validate_scaled = np.concat((np.ones(X_validate_scaled.shape[0]).reshape(-1, 1), X_validate_scaled), axis=1)

  return X_train_standard, X_validate_scaled, y_train, y_validate


In [6]:
features = []
results = []

X_train, X_validate, y_train, y_validate = produce_training_data(df)
all_features = df.columns.drop(['tip', 'day'])

for feature in all_features:
  features.append(feature)
  X_train_new = X_train[:, :len(features) + 1]
  X_validate_new = X_validate[:, :len(features) + 1]

  weights = ols(X_train_new, y_train)
  pred_ols = X_validate_new @ weights

  results.append({
    "MSE": mse(y_validate, pred_ols),
    "RMSE": rmse(y_validate, pred_ols), 
    "MAE": mae(y_validate, pred_ols), 
    "R2": r_squared(y_validate, pred_ols), 
    "Adjusted_R2": adjusted_r_squared(y_validate, pred_ols, X_validate_new.shape[1] - 1),
    "Features": " | ".join(features),
  })


pd.DataFrame(results)

,MSE,RMSE,MAE,R2,Adjusted_R2,Features
0,1.390201,1.179068,0.854350,0.002506,-0.024453,total_bill
1,1.391072,1.179437,0.854713,0.001881,-0.053570,total_bill | sex
2,1.381272,1.175275,0.853806,0.008913,-0.076037,total_bill | sex | smoker
3,1.390328,1.179122,0.862753,0.002415,-0.114948,total_bill | sex | smoker | time
4,1.334285,1.155113,0.861631,0.042627,-0.102430,total_bill | sex | smoker | time | size
